# 03 - Rolling-window training

Step 3 of `THESIS_GUIDE.md`. The six models are trained on the FinBench rolling protocol (4 years of training, 1 of validation, 1 of test; test years 2020-2024) in three universes (DJI, NASDAQ-100, EURO STOXX 50), with T = 20 and L = 5. Each model is trained **exactly as FinBench trains it** and keeps its weights with FinBench's own rule (decided 2026-10-06, `xaifin.training.trainer.SELECTION_RULES`); the data-quality filter is on.

This notebook reads the results store (`xaifin.training.rolling.collect_results`) and answers three questions:

1. **How well do the models predict?** IC, RankIC, ICIR and MSE per model, universe and test year.
2. **Are the numbers in FinBench's range?** FinBench publishes only MSE and MAE for these models (`results/analysis/finbench_table2.csv`).
3. **How weak is the signal?** The starting point of the thesis: if the predictions barely rank the stocks, explanations and portfolios built on them can be unstable.

Metrics are computed on the test year with the labels the model is scored on (the daily z-score of the forward return). **IC** and **RankIC** are the daily cross-sectional Pearson and Spearman correlations between predictions and labels, averaged over the test days; **ICIR** and **RankICIR** divide that mean by the standard deviation of the daily values. FactorVAE's label is a 20-day return (issue 3 in `docs/model_notes.md`), and FinFormer's loss ignores scale, so its MSE is not comparable (notebook 02).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TEST_YEARS, UNIVERSE_NAMES
from xaifin.training.rolling import collect_results

SEQ_LEN, PRED_LEN = 20, 5
MODELS = viz.MODEL_ORDER
all_runs = collect_results()
runs = all_runs[(all_runs["seq_len"] == SEQ_LEN) & (all_runs["pred_len"] == PRED_LEN) & all_runs["universe"].isin(CORE_UNIVERSES)]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"{len(runs)} finished runs (T={SEQ_LEN}, L={PRED_LEN}), seeds {sorted(runs['seed'].unique())}")

84 finished runs (T=20, L=5), seeds [np.int64(42)]


## 1. The grid

One run per model, universe, test year and seed. EURO STOXX 50 has no 2020 run: its membership history starts on 2020-09-30, so no stock is a member at the start of 2020 (notebook 01). Training time is on one RTX 5090; FactorVAE is the slow one (FinBench's 96 attention layers run one after another for every day).

In [2]:
expected = {u: [y for y in TEST_YEARS if not (u == "sx5e" and y == 2020)] for u in CORE_UNIVERSES}
coverage = (runs.groupby(["model", "universe"]).size().unstack("universe").reindex(index=MODELS, columns=CORE_UNIVERSES)
            .fillna(0).astype(int))
coverage.loc["expected per model"] = [len(expected[u]) * runs["seed"].nunique() for u in CORE_UNIVERSES]
display(coverage)
timing = runs.groupby("model").agg(runs=("minutes", "size"), minutes_mean=("minutes", "mean"), minutes_total=("minutes", "sum"),
                                   epochs_run=("epochs_run", "mean"), kept_epoch=("kept_epoch", "mean")).reindex(MODELS)
timing

universe,dji,nasdaq100,sx5e
model,,,
MASTER,5,5,4
MATCC,5,5,4
FactorVAE,5,5,4
HIST,5,5,4
DiscoverPLF,5,5,4
FinFormer,5,5,4
expected per model,5,5,4


,runs,minutes_mean,minutes_total,epochs_run,kept_epoch
model,,,,,
MASTER,14,1.9071,26.7000,11.2143,10.2143
MATCC,14,15.9071,222.7000,70.0000,69.0000
FactorVAE,14,66.0571,924.8000,30.0000,0.8571
HIST,14,10.7571,150.6000,54.6429,23.6429
DiscoverPLF,14,23.9000,334.6000,48.5000,28.2857
FinFormer,14,5.1071,71.5000,16.6429,5.6429


## 2. Test performance per model, universe and year

Mean over the seeds trained so far. Blue cells: the predictions rank the stocks in the right direction on average over the test year; red: the wrong direction.

In [3]:
cells = runs.groupby(["universe", "model", "test_year"])[["IC", "RankIC", "ICIR", "RankICIR", "MSE", "MAE"]].mean().reset_index()
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if grid.empty:
        continue
    viz.metric_heatmap(grid, "RankIC", f"{UNIVERSE_NAMES[universe]}: test RankIC by model and test year").show()
    display(grid.pivot(index="model", columns="test_year", values="RankIC").reindex([m for m in MODELS if m in set(grid["model"])]))

test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0529,-0.0109,-0.0002,0.0060,0.0201
MATCC,0.0248,0.0025,0.0309,0.0321,-0.0238
FactorVAE,0.0501,0.0963,-0.0479,0.0711,-0.0269
HIST,0.0365,0.0193,-0.0034,0.0229,0.0705
DiscoverPLF,0.0087,0.0192,0.0176,0.0331,0.0081
FinFormer,0.0301,0.0409,0.0129,-0.0035,0.0188


test_year,2020,2021,2022,2023,2024
model,,,,,
MASTER,0.0390,0.0110,0.0102,0.0070,0.0091
MATCC,0.0488,0.0277,-0.0305,0.0189,0.0195
FactorVAE,-0.0044,-0.0784,-0.0727,0.0062,-0.0242
HIST,0.0215,0.0163,0.0011,0.0100,0.0166
DiscoverPLF,0.0168,-0.0156,-0.0173,0.0061,0.0116
FinFormer,0.0479,0.0042,-0.0151,-0.0041,-0.0187


test_year,2021,2022,2023,2024
model,,,,
MASTER,0.0214,-0.0129,-0.0173,-0.0275
MATCC,0.0002,-0.0251,-0.0076,-0.0074
FactorVAE,0.0367,-0.0069,-0.0377,-0.0412
HIST,-0.0100,0.0074,0.0049,-0.0322
DiscoverPLF,0.0175,0.0135,0.0172,-0.0486
FinFormer,0.0294,-0.0212,0.0100,-0.0202


In [4]:
for universe in CORE_UNIVERSES:
    grid = cells[cells["universe"] == universe]
    if not grid.empty:
        viz.metric_by_year_chart(grid, "IC", f"{UNIVERSE_NAMES[universe]}: test IC per year").show()

## 3. Averages over the test years, and FinBench's numbers

Per model and universe: the mean over test years (and seeds) of each metric, and its standard deviation across years. FinBench's Table 2 reports MSE and MAE averaged over its rolling test years and seeds; its FactorVAE numbers come from the leaky readout (issue 1), and its MASTER, MATCC and FinFormer runs were not seeded (issue 8).

In [5]:
finbench = pd.read_csv(RESULTS_ROOT / "analysis" / "finbench_table2.csv").query("pred_len == @PRED_LEN")
summary = (runs.groupby(["universe", "model"])
           .agg(years=("test_year", "nunique"), IC=("IC", "mean"), IC_std=("IC", "std"), RankIC=("RankIC", "mean"),
                RankIC_std=("RankIC", "std"), ICIR=("ICIR", "mean"), RankICIR=("RankICIR", "mean"),
                MSE=("MSE", "mean"), MAE=("MAE", "mean"))
           .reset_index()
           .merge(finbench.rename(columns={"MSE": "FinBench MSE", "MAE": "FinBench MAE"})[["universe", "model", "FinBench MSE", "FinBench MAE"]],
                  on=["universe", "model"], how="left"))
summary["model"] = pd.Categorical(summary["model"], MODELS)
summary = summary.sort_values(["universe", "model"]).reset_index(drop=True)
summary

,universe,model,years,IC,IC_std,RankIC,RankIC_std,ICIR,RankICIR,MSE,MAE,FinBench MSE,FinBench MAE
0,dji,MASTER,5,0.0158,0.0199,0.0136,0.0247,0.0713,0.0608,0.9806,0.7462,1.2490,0.8590
1,dji,MATCC,5,0.0072,0.0302,0.0133,0.0239,0.0298,0.0603,1.4889,0.9386,1.1300,0.8180
2,dji,FactorVAE,5,0.0270,0.0692,0.0285,0.0628,0.1420,0.1423,0.8880,0.7242,0.9710,0.7580
3,dji,HIST,5,0.0288,0.0275,0.0292,0.0272,0.1239,0.1328,0.9706,0.7421,0.9970,0.7470
4,dji,DiscoverPLF,5,0.0160,0.0170,0.0174,0.0102,0.0727,0.0795,1.1119,0.8003,1.2110,0.8300
5,dji,FinFormer,5,0.0122,0.0194,0.0199,0.0169,0.0464,0.0819,2.9037,1.3697,1.6570,0.9820
6,nasdaq100,MASTER,5,0.0111,0.0124,0.0152,0.0133,0.0575,0.0859,1.0139,0.7170,1.0170,0.7200
7,nasdaq100,MATCC,5,0.0111,0.0228,0.0169,0.0291,0.0806,0.1236,1.7238,0.9873,1.7130,0.9810
8,nasdaq100,FactorVAE,5,-0.0341,0.0406,-0.0347,0.0389,-0.1965,-0.1750,0.9041,0.7188,0.9180,0.7250
9,nasdaq100,HIST,5,0.0109,0.0141,0.0131,0.0079,0.0538,0.0683,1.0184,0.7171,1.0300,0.7210


## 4. How weak is the signal?

Three views of the same question. (a) The share of runs whose test RankIC is above 0: a model with no skill lands there half of the time. (b) The t-statistic of the mean daily RankIC. The labels of consecutive days overlap (each is a 5-day return), so the daily values are correlated and RankICIR × √(test days) would overstate it; RankICIR × √(test days / L) counts only non-overlapping periods. About 2 or more means the year's mean is unlikely to be noise. (c) The best test RankIC per universe and year: the ceiling of what any of the six reaches.

In [6]:
days = 252  # test days per year
weak = runs.assign(t_stat=runs["RankICIR"] * np.sqrt(days / PRED_LEN))
share = (weak.groupby("model").agg(runs=("RankIC", "size"), positive_RankIC=("RankIC", lambda s: (s > 0).mean()),
                                   mean_RankIC=("RankIC", "mean"), significant=("t_stat", lambda s: (s.abs() >= 2).mean()))
         .reindex(MODELS))
display(share.style.format({"positive_RankIC": "{:.0%}", "significant": "{:.0%}", "mean_RankIC": "{:+.4f}"}))
best = weak.loc[weak.groupby(["universe", "test_year"])["RankIC"].idxmax(), ["universe", "test_year", "model", "RankIC", "t_stat"]]
best

,runs,positive_RankIC,mean_RankIC,significant
model,,,,
MASTER,14,64%,+0.0077,0%
MATCC,14,64%,+0.0079,7%
FactorVAE,14,36%,-0.0057,29%
HIST,14,79%,+0.0130,7%
DiscoverPLF,14,79%,+0.0063,0%
FinFormer,14,57%,+0.0080,0%


,universe,test_year,model,RankIC,t_stat
56,dji,2020,MASTER,0.0529,1.5343
15,dji,2021,FactorVAE,0.0963,3.2858
72,dji,2022,MATCC,0.0309,1.0247
17,dji,2023,FactorVAE,0.0711,2.8780
46,dji,2024,HIST,0.0705,2.4835
75,nasdaq100,2020,MATCC,0.0488,2.1147
76,nasdaq100,2021,MATCC,0.0277,1.3263
63,nasdaq100,2022,MASTER,0.0102,0.3001
78,nasdaq100,2023,MATCC,0.0189,1.0432
79,nasdaq100,2024,MATCC,0.0195,1.0717


## 5. The selection rules

Each model keeps its weights with FinBench's rule. If the validation score that picks the weights carried information about the test year, the kept epoch's validation IC and the test IC would be related; with a weak signal they need not be. The table shows, per model, the correlation across runs between the validation IC of the kept epoch and the test IC, and how many epochs the rule ran and kept.

In [7]:
selection = (runs.groupby("model")
             .apply(lambda d: pd.Series({"runs": len(d), "valid_IC_mean": d["valid_IC"].mean(), "test_IC_mean": d["IC"].mean(),
                                         "corr(valid IC, test IC)": d["valid_IC"].corr(d["IC"]) if len(d) > 2 else np.nan,
                                         "epochs_run": d["epochs_run"].mean(), "kept_epoch": d["kept_epoch"].mean()}),
                    include_groups=False)
             .reindex(MODELS))
selection

,runs,valid_IC_mean,test_IC_mean,"corr(valid IC, test IC)",epochs_run,kept_epoch
model,,,,,,
MASTER,14.0000,0.0087,0.0084,0.0354,11.2143,10.2143
MATCC,14.0000,0.0128,0.0053,-0.0365,70.0000,69.0000
FactorVAE,14.0000,0.0148,-0.0056,-0.1571,30.0000,0.8571
HIST,14.0000,0.0345,0.0120,0.2761,54.6429,23.6429
DiscoverPLF,14.0000,0.0234,0.0065,-0.0417,48.5000,28.2857
FinFormer,14.0000,0.0250,0.0052,0.1438,16.6429,5.6429


## 6. Saved summary

`results/analysis/metrics_summary.csv`: one row per run (cell, test metrics, kept epoch, epochs run, minutes, validation IC of the kept epoch). Later steps and the thesis tables read it.

In [8]:
out = RESULTS_ROOT / "analysis" / "metrics_summary.csv"
all_runs.drop(columns="path").sort_values(["model", "universe", "seq_len", "pred_len", "test_year", "seed"]).to_csv(out, index=False)
print(f"{len(all_runs)} runs -> {out}")

84 runs -> D:\CodeProjects\master-thesis\code\results\analysis\metrics_summary.csv


## Observations

Stage 3 of the grid: 6 models x 3 universes x test years 2020-2024 (EURO STOXX 50 from 2021), seed 42, T = 20, L = 5, data-quality filter on.

1. **The grid is complete: 84 runs**, about 29 hours of GPU time on one RTX 5090. FactorVAE alone took 15 hours (66 minutes per run, FinBench's sequential attention layers); MASTER 2 minutes per run, FinFormer 5, HIST 11, MATCC 16, DiscoverPLF 24.
2. **The signal is weak everywhere.** Averaged over its 14 runs, each model's test RankIC is between -0.006 (FactorVAE) and +0.013 (HIST). RankIC is above 0 in 36% (FactorVAE) to 79% (HIST, DiscoverPLF) of the runs, where a model with no skill would land at about 50%. Counting only non-overlapping weeks, a run's mean RankIC is clearly different from 0 (|t| >= 2) in at most 1 of 14 runs for every model except FactorVAE (4 of 14, in both directions).
3. **It depends on the universe more than on the model.** On DJI every model has a positive average IC (0.007 to 0.029); on NASDAQ-100 averages are 0 to 0.011, and FactorVAE's is negative (-0.034); on EURO STOXX 50 all six are at or below 0 (-0.011 to +0.003).
4. **No model wins consistently.** The best model changes from year to year and universe to universe (section 4, last table): FactorVAE, MATCC, MASTER, HIST and DiscoverPLF all top at least one universe-year, and the best RankIC of a cell ranges from -0.007 to 0.096.
5. **MSE is in FinBench's range** for MASTER (NASDAQ-100, EURO STOXX 50), FactorVAE, HIST and DiscoverPLF, within about 0.1 of FinBench's Table 2; MATCC's and FinFormer's are higher (MATCC 1.49-1.72, FinFormer 2.2-2.9; FinFormer's loss ignores scale), and MASTER's on DJI is lower (0.98 vs 1.25). FinBench's numbers average its own runs, which were partly unseeded and, for FactorVAE, scored differently, so only the range is comparable.
6. **The selection rules choose almost blindly.** Across runs, the validation IC of the kept epoch and the test IC are barely related (correlation -0.16 to 0.28 by model). FactorVAE's rule (lowest validation loss) keeps epoch 0 or 1 of 30 in almost every run: a barely trained model. Validation scores are optimistic: the kept epochs average a validation IC of 0.009-0.035 and a test IC of -0.006 to 0.012.

Starting point for the thesis: the six architectures extract a small, unstable signal whose sign changes with the period and the universe. Whether their explanations agree, and whether disagreement comes from the data or the models, is the question of Steps 5-7.